In [ ]:
import os 
import re 
import sys
import argparse
import numpy as np
import pandas as pd
from glob import glob 
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import networkx as nx 
from networkx.drawing.nx_agraph import graphviz_layout
from itertools import combinations

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 40)
pd.options.display.float_format = '{:.2f}'.format

PATIENT = 'PPCG0086'
# INDIR_DPCLUST = '/home/grace/work/PPCG_DifferentialGenesetMutation/data/angel/dpclust_ccf'
# INDIR_SCNA = '/home/grace/work/PPCG_DifferentialGenesetMutation/data/angel/06.03.2026/battenberg_scna/Subclonal_SCNA_with_Avg_CN'
INDIR_DPCLUST = '/home/grace/work/PPCG_DifferentialGenesetMutation/data/angel/26.03.2026/ccf/Clonal_trees'
INDIR_TREES = '/home/grace/work/PPCG_DifferentialGenesetMutation/data/angel/06.03.2026/conipher_trees'
INDIR_SCNA = '/home/grace/work/PPCG_DifferentialGenesetMutation/data/angel/26.03.2026/Subclonal_SCNA_with_Avg_CN'
INFILE_SHEET = '/home/grace/work/PPCG_DifferentialGenesetMutation/samplesheet.angel.alldonors.tsv'
INFILE_COSMIC = '/home/grace/work/PPCG_DifferentialGenesetMutation/data/supporting/cosmic/Cosmic_Genes_v103_GRCh37.tsv'
# INFILE_HGNC = '/home/grace/work/PPCG_DifferentialGenesetMutation/data/supporting/hgnc/hgnc_complete_set.txt'
INFILE_GENCODE = '/home/grace/work/PPCG_DifferentialGenesetMutation/data/supporting/gencode/gencode.v31lift37.basic.annotation.gff3'
OUTDIR = '/home/grace/work/PPCG_DifferentialGenesetMutation/results/CNA'

CHRSIZES = {
    '1': 249_250_621, '12': 133_851_895,
    '2': 243_199_373, '13': 115_169_878,
    '3': 198_022_430, '14': 107_349_540,
    '4': 191_154_276, '15': 102_531_392,
    '5': 180_915_260, '16': 90_354_753,
    '6': 171_115_067, '17': 81_195_210,
    '7': 159_138_663, '18': 78_077_248,
    '8': 146_364_022, '19': 59_128_983,
    '9': 141_213_431, '20': 63_025_520,
    '10': 135_534_747, '21': 48_129_895,
    '11': 135_006_516, '22': 51_304_566,
    'X': 155_270_560,
}

sheet = pd.read_csv(INFILE_SHEET, sep='\t', header=0)
donors_dpc = set([x.split('/')[-1][:8] for x in glob(f"{INDIR_DPCLUST}/*_Cluster_CCFs.csv")])
donors_tree = set([x.split('/')[-2][:8] for x in glob(f"{INDIR_TREES}/*/allTrees.txt")])
donors_primary = set(sheet[sheet['tissue']=='Primary']['donor'].unique())
ALL_DONORS = sorted(list(donors_dpc & donors_tree))
SAMPLE_2_WGD_LUT = sheet.set_index('sample')['WGD'].to_dict()



In [ ]:
def load_cosmic_genelist(filepath: str) -> list[str]:
    cosmic = pd.read_csv(filepath, sep='\t', header=0)
    cosmic = cosmic[cosmic['IN_CANCER_CENSUS']=='y']
    cosmic['HGNC_ID'] = cosmic['HGNC_ID'].apply(lambda x: str(int(x))).astype(str)
    return list(set(cosmic['GENE_SYMBOL'].unique()))

def load_gff_gencode(filepath: str, allow_noncoding: bool=False, allow_x: bool=False, allow_y: bool=False) -> pd.DataFrame:

    def get_name(text: str) -> str:
        PATTERN = r'^.*?gene_name=([A-Za-z0-9_\.-]+).*?$'
        m = re.match(PATTERN, text)
        assert m is not None
        return m.group(1)

    def get_gtype(text: str) -> str:
        PATTERN = r'^.*?gene_type=([A-Za-z0-9_-]+).*?$'
        m = re.match(PATTERN, text)
        assert m is not None
        return m.group(1)
    
    def get_hgnc_id(text: str) -> str|None:
        PATTERN = r'^.*?hgnc_id=HGNC:([^;]+).*?$'
        m = re.match(PATTERN, text)
        if m is not None:
            return m.group(1)
        return None 

    data = []
    with open(filepath, 'r') as fp:
        line = fp.readline().strip('\n')
        i = 0
        while line:
            if line.startswith('#'):
                line = fp.readline().strip('\n')
                continue 
            
            i += 1
            if i % 10_000 == 0:
                print(f"processed {i} records...", end='\r')
            
            lsplit = line.split('\t')
            chrom = lsplit[0].replace('chr', '')

            if lsplit[2] != 'gene':
                line = fp.readline().strip('\n')
                continue
            if chrom == 'X' and not allow_x:
                line = fp.readline().strip('\n')
                continue 
            if chrom == 'Y' and not allow_y:
                line = fp.readline().strip('\n')
                continue 
            
            start = int(lsplit[3])
            end = int(lsplit[4])
            gene = get_name(lsplit[8])
            hgnc_id = get_hgnc_id(lsplit[8])
            gtype = get_gtype(lsplit[8])

            if gtype == 'protein_coding' or allow_noncoding:
                # data.append((chrom, start, end, gene, hgnc_id))
                data.append((chrom, start, end, gene))

            line = fp.readline().strip('\n')

    print(f"processed {i} records... done")
    # df = pd.DataFrame.from_records(data, columns=['chr', 'start', 'end', 'gene', 'HGNC_ID'])
    df = pd.DataFrame.from_records(data, columns=['chr', 'start', 'end', 'gene'])
    df['chr'] = df['chr'].astype(str)
    df['start'] = df['start'].astype(int)
    df['end'] = df['end'].astype(int)
    df['gene'] = df['gene'].astype(str)
    # df['HGNC_ID'] = df['HGNC_ID'].astype(str)
    return df

cosmic_genes = load_cosmic_genelist(INFILE_COSMIC)
gencode = load_gff_gencode(INFILE_GENCODE, allow_noncoding=True, allow_x=True, allow_y=False)
gencode = gencode[gencode['gene'].isin(set(cosmic_genes))].copy()

data = [
    ('14', 22_090_057, 23_021_075, 'TRA'), 
    ('22', 22_380_474, 23_265_085, 'IGL'), 
    ('14', 22_891_537, 22_935_569, 'TRD'), 
    ('17', 8_043_790, 8_059_824, 'PER1'), 
    ('2', 89_156_874, 90_274_235, 'IGK'), 
    ('7', 141_998_851, 142_510_972, 'TRB'), 
    ('17', 78_234_651, 78_372_594, 'RNF213'), 
    ('14', 106_052_774, 107_288_051, 'IGH'), 
    ('17', 9_813_923, 10_101_923, 'GAS7'), 
    ('18', 23_596_217, 23_671_183, 'SS18')
]
extras = pd.DataFrame.from_records(data, columns=['chr', 'start', 'end', 'gene'])
gencode = pd.concat([gencode, extras], ignore_index=True)
print(gencode['gene'].nunique())
print(gencode.shape[0])
outfile = '/home/grace/work/PPCG_DifferentialGenesetMutation/data/supporting/gencode/gencode.genelocations.tsv'
gencode.to_csv(outfile, sep='\t', index=False)
gencode.head()
